# TIER.0) Load scored universe (single source of truth)

In [ ]:
# ============================================================
# TIER.0) Load scored universe (single source of truth)
# - Input: artifacts/eval_universe/eval_scored_DG_V3.parquet
# - Output: eval_base (row-level grain: NPI x HCPCS x POS x Year)
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

EVAL_PATH = Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")
assert EVAL_PATH.exists(), f"Missing: {EVAL_PATH}"

eval_base = pd.read_parquet(EVAL_PATH)

print("Loaded eval_base:", eval_base.shape)
print("Columns:", len(eval_base.columns))

# Basic required columns for tiering
REQ = [
    "row_id",
    "Rndrng_NPI", "provider_type", "state",
    "HCPCS_Cd", "Year",
    "Place_Of_Srvc",
    "observed_cost", "expected_cost", "residual", "oe_ratio", "log_oe",
    "services", "benes",
    "expected_cost_support_tier",
    "high_confidence_anomaly_candidate",
]
missing = [c for c in REQ if c not in eval_base.columns]
if missing:
    raise KeyError(f"eval_scored_DG_V3 missing required columns for tiering: {missing}")

# Optional sanity on grain (should be unique at (NPI,HCPCS,POS,Year))
keys = ["Rndrng_NPI","HCPCS_Cd","Place_Of_Srvc","Year"]
n_dupe = int(eval_base.duplicated(keys).sum())
print("Duplicate key rows:", n_dupe)
if n_dupe:
    display(eval_base.loc[eval_base.duplicated(keys, keep=False), keys + ["row_id"]].head(20))

# TIER.1) Minimal anomaly feature layer for tiering

In [ ]:
# ============================================================
# TIER.1) Minimal anomaly feature layer for tiering
# Builds:
#   - is_high_conf
#   - log_oe_pct_in_slice within (HCPCS_Cd, Year)
#   - slice_n within (HCPCS_Cd, Year)
# Notes:
#   - Keeps definitions aligned with anomaly surfacing notebook.
# ============================================================

import numpy as np
import pandas as pd

df = eval_base.copy()

# -----------------------------
# High-confidence definition (same knobs as ANOM.1)
# -----------------------------
MIN_SERVICES = 50
MIN_BENES = 20
HIGH_CONF_TIERS = {"high", "medium_high"}

df["is_high_conf"] = (
    df["expected_cost_support_tier"].astype(str).isin(HIGH_CONF_TIERS)
    & (df["services"].fillna(0) >= MIN_SERVICES)
    & (df["benes"].fillna(0) >= MIN_BENES)
)

# Keep existing candidate flag (force bool)
df["high_confidence_anomaly_candidate"] = df["high_confidence_anomaly_candidate"].astype(bool)

# -----------------------------
# Slice features: (HCPCS_Cd, Year)
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")

eval_tier = df
print("Defined eval_tier:", eval_tier.shape)

# Quick sanity
print("is_high_conf true %:", float(eval_tier["is_high_conf"].mean() * 100))
print("slice_n min/median/p1/p99/max:",
      int(eval_tier["slice_n"].min()),
      float(eval_tier["slice_n"].median()),
      float(eval_tier["slice_n"].quantile(0.01)),
      float(eval_tier["slice_n"].quantile(0.99)),
      int(eval_tier["slice_n"].max()))

# TIER.2) Provider summaries (robust repeat offenders + shock severity), *NO `TOP_N` trimming*

In [ ]:
# ============================================================
# TIER.2) Provider summaries (robust repeat offenders + shock severity), NO TOP_N trimming
# Produces:
#   - provider_summary_robust_sizeaware_all
#   - provider_summary_mag_sizeaware_all
# Notes:
#   - Uses same event definitions as ANOM.3.a.1 and ANOM.3.b.1
#   - Keeps provider grain: (Rndrng_NPI, provider_type, state)
# ============================================================

import numpy as np
import pandas as pd

df = eval_tier.copy()

# -----------------------------
# Shared knobs (tiering)
# -----------------------------
MIN_SLICE_N = 50

USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2

# For shock severity cutoff
LOG_OE_Q = 0.995

# Ensure numeric
for c in ["log_oe","residual","expected_cost","services","benes"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Optional denom hygiene (recommended)
if USE_MIN_EXPECTED_FILTER:
    df = df[df["expected_cost"].notna() & (df["expected_cost"] >= MIN_EXPECTED)].copy()

# -----------------------------
# ROBUST extreme-event flag (repeat offenders): ANOM.3.a.1 definition
# -----------------------------
df["is_row_anomalous_robust_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["slice_n"] >= MIN_SLICE_N)
)

# Provider summary (robust)
GROUP = ["Rndrng_NPI","provider_type","state"]
provider_summary_robust_sizeaware_all = (
    df.groupby(GROUP, dropna=False)
      .agg(
          n_rows=("row_id","size"),
          n_anom_rows_robust=("is_row_anomalous_robust_sizeaware","sum"),
          anom_rate_pct_robust=("is_row_anomalous_robust_sizeaware", lambda s: float(s.mean()*100)),
          n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
          n_unique_years=("Year", pd.Series.nunique),
          median_log_oe=("log_oe","median"),
          median_residual=("residual","median"),
          p75_log_oe=("log_oe", lambda s: float(np.nanpercentile(s, 75))),
          p90_log_oe=("log_oe", lambda s: float(np.nanpercentile(s, 90))),
          p75_residual=("residual", lambda s: float(np.nanpercentile(s, 75))),
          p90_residual=("residual", lambda s: float(np.nanpercentile(s, 90))),
          total_services=("services","sum"),
          total_benes=("benes","sum"),
          pct_high_conf_rows=("is_high_conf", lambda s: float(s.mean()*100)),
      )
      .reset_index()
)

# Optional: keep providers with >0 events as a field, but do NOT filter here by default
provider_summary_robust_sizeaware_all["provider_anom_score_robust"] = (
    provider_summary_robust_sizeaware_all["n_anom_rows_robust"]
    + 0.25 * provider_summary_robust_sizeaware_all["n_unique_codes"]
    + 0.25 * provider_summary_robust_sizeaware_all["n_unique_years"]
)

print("provider_summary_robust_sizeaware_all:", provider_summary_robust_sizeaware_all.shape)

# -----------------------------
# MAG shock-event flag: ANOM.3.b.1 definition
# - severity cutoff computed on BASE (log_oe>0 & residual>0) after denom hygiene
# -----------------------------
BASE = df[(df["log_oe"] > 0) & (df["residual"] > 0)].copy()
if len(BASE) == 0:
    raise ValueError("No BASE rows found for severity cut (log_oe>0 & residual>0). Check inputs/filters.")

log_oe_severity_cut = float(BASE["log_oe"].quantile(LOG_OE_Q))
print(f"log_oe_severity_cut (q={LOG_OE_Q}): {log_oe_severity_cut:.6f} (oe_ratio ~ {np.exp(log_oe_severity_cut):.2f}x)")

df["is_row_anomalous_mag_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["log_oe"] >= log_oe_severity_cut)
    & (df["slice_n"] >= MIN_SLICE_N)
)

def _masked_max(s: pd.Series, mask: pd.Series) -> float:
    x = s[mask]
    return float(x.max()) if len(x) else np.nan

def _masked_p95(s: pd.Series, mask: pd.Series) -> float:
    x = s[mask]
    return float(np.nanpercentile(x, 95)) if len(x) else np.nan

# Provider summary (mag) computed on anomalous rows for severity stats
provider_summary_mag_sizeaware_all = (
    df.groupby(GROUP, dropna=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_anom_rows_mag": int(g["is_row_anomalous_mag_sizeaware"].sum()),
          "anom_rate_pct_mag": float(g["is_row_anomalous_mag_sizeaware"].mean() * 100),
          "n_unique_codes": int(g["HCPCS_Cd"].nunique()),
          "n_unique_years": int(g["Year"].nunique()),
          "max_log_oe_mag": _masked_max(g["log_oe"], g["is_row_anomalous_mag_sizeaware"]),
          "p95_log_oe_mag": _masked_p95(g["log_oe"], g["is_row_anomalous_mag_sizeaware"]),
          "median_log_oe": float(g["log_oe"].median()),
          "total_services": float(np.nansum(g["services"].to_numpy())),
          "total_benes": float(np.nansum(g["benes"].to_numpy())),
      }))
      .reset_index()
)

provider_summary_mag_sizeaware_all["provider_anom_score_mag"] = (
    provider_summary_mag_sizeaware_all["n_anom_rows_mag"]
    + 0.25 * provider_summary_mag_sizeaware_all["n_unique_codes"]
    + 0.25 * provider_summary_mag_sizeaware_all["n_unique_years"]
    + 0.10 * provider_summary_mag_sizeaware_all["p95_log_oe_mag"].fillna(0)
)

print("provider_summary_mag_sizeaware_all:", provider_summary_mag_sizeaware_all.shape)

# Optional quick sanity: how many providers have >=1 event in each definition
print("Providers with >=1 robust event:",
      int((provider_summary_robust_sizeaware_all["n_anom_rows_robust"] > 0).sum()))
print("Providers with >=1 shock event:",
      int((provider_summary_mag_sizeaware_all["n_anom_rows_mag"] > 0).sum()))

# TIER.3) Build provider_scorecard_v1 + freeze to artifacts/provider_tiering/

In [ ]:
# ============================================================
# TIER.3) Build provider_scorecard_v1 + freeze to artifacts/provider_tiering/
# Output:
#   - provider_scorecard_v1__{ts}.parquet
#   - provider_scorecard_v1__{ts}.csv
#   - params__{ts}.json + params__{ts}.md + manifest__{ts}.json
# Notes:
#   - Keep ALL providers. Downstream tiering can filter by volume / n_rows.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

# -----------------------------
# 0) Join robust + mag summaries into one scorecard
# -----------------------------
KEY = ["Rndrng_NPI","provider_type","state"]

rob = provider_summary_robust_sizeaware_all.copy()
mag = provider_summary_mag_sizeaware_all.copy()

for c in KEY:
    if c not in rob.columns or c not in mag.columns:
        raise KeyError(f"Missing {KEY} in provider summaries.")

provider_scorecard_v1 = rob.merge(
    mag[KEY + [c for c in mag.columns if c not in KEY]],
    on=KEY,
    how="left",
    suffixes=("_rob","_mag"),
)

# Fill missing mag metrics (providers with no shock events)
for c in ["n_anom_rows_mag","anom_rate_pct_mag","max_log_oe_mag","p95_log_oe_mag","provider_anom_score_mag"]:
    if c in provider_scorecard_v1.columns:
        provider_scorecard_v1[c] = pd.to_numeric(provider_scorecard_v1[c], errors="coerce").fillna(0)

# Helpful derived fields (optional)
provider_scorecard_v1["has_any_robust_event"] = provider_scorecard_v1["n_anom_rows_robust"] > 0
provider_scorecard_v1["has_any_shock_event"] = provider_scorecard_v1["n_anom_rows_mag"] > 0

print("provider_scorecard_v1:", provider_scorecard_v1.shape)
display(provider_scorecard_v1.head(5))

# -----------------------------
# 1) Freeze to disk with params
# -----------------------------
ts = pd.Timestamp.now(tz="America/New_York").strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("artifacts/provider_tiering") / f"run_{ts}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = OUT_DIR / f"provider_scorecard_v1__{ts}.parquet"
CSV_PATH     = OUT_DIR / f"provider_scorecard_v1__{ts}.csv"

provider_scorecard_v1.to_parquet(PARQUET_PATH, index=False)
provider_scorecard_v1.to_csv(CSV_PATH, index=False)

PARAMS = {
    "timestamp_et": ts,
    "outputs_dir": str(OUT_DIR),
    "input_eval_path": str(Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")),
    "provider_grain": KEY,
    "tier_feature_layer": {
        "MIN_SERVICES": MIN_SERVICES,
        "MIN_BENES": MIN_BENES,
        "HIGH_CONF_TIERS": sorted(list(HIGH_CONF_TIERS)),
        "slice_cols": ["HCPCS_Cd","Year"],
    },
    "robust_events": {
        "MIN_SLICE_N": MIN_SLICE_N,
        "USE_MIN_EXPECTED_FILTER": USE_MIN_EXPECTED_FILTER,
        "MIN_EXPECTED": MIN_EXPECTED if USE_MIN_EXPECTED_FILTER else None,
        "event_flag": "(log_oe>0) & (residual>0) & (is_high_conf | high_confidence_anomaly_candidate) & (log_oe_pct_in_slice>=0.99) & (slice_n>=MIN_SLICE_N)",
        "provider_score": "n_anom_rows_robust + 0.25*n_unique_codes + 0.25*n_unique_years",
    },
    "shock_events": {
        "MIN_SLICE_N": MIN_SLICE_N,
        "USE_MIN_EXPECTED_FILTER": USE_MIN_EXPECTED_FILTER,
        "MIN_EXPECTED": MIN_EXPECTED if USE_MIN_EXPECTED_FILTER else None,
        "LOG_OE_Q": LOG_OE_Q,
        "log_oe_severity_cut": float(log_oe_severity_cut),
        "event_flag": "(log_oe>0) & (residual>0) & (is_high_conf | high_confidence_anomaly_candidate) & (log_oe_pct_in_slice>=0.99) & (log_oe>=log_oe_severity_cut) & (slice_n>=MIN_SLICE_N)",
        "provider_score": "n_anom_rows_mag + 0.25*n_unique_codes + 0.25*n_unique_years + 0.10*p95_log_oe_mag",
    },
    "files": {
        "parquet": str(PARQUET_PATH),
        "csv": str(CSV_PATH),
    },
}

PARAMS_JSON = OUT_DIR / f"params__{ts}.json"
MANIFEST_JSON = OUT_DIR / f"manifest__{ts}.json"
PARAMS_MD = OUT_DIR / f"params__{ts}.md"

with open(PARAMS_JSON, "w") as f:
    json.dump(PARAMS, f, indent=2)

with open(MANIFEST_JSON, "w") as f:
    json.dump(
        {"timestamp_et": ts, "outputs_dir": str(OUT_DIR),
         "artifacts": [{"name":"provider_scorecard_v1","rows":int(len(provider_scorecard_v1)),"cols":int(provider_scorecard_v1.shape[1]),
                        "parquet": str(PARQUET_PATH), "csv": str(CSV_PATH)}]},
        f, indent=2
    )

md = []
md.append(f"# Provider tiering scorecard (v1) — {ts} ET\n\n")
md.append(f"Output directory: `{OUT_DIR}`\n\n")
md.append("## Artifacts\n")
md.append(f"- Parquet: `{PARQUET_PATH}`\n")
md.append(f"- CSV: `{CSV_PATH}`\n\n")
md.append("## Parameters\n")
md.append("```json\n" + json.dumps(PARAMS, indent=2) + "\n```\n")
PARAMS_MD.write_text("".join(md))

print("✅ Wrote provider_scorecard_v1")
print(" -", PARQUET_PATH)
print(" -", CSV_PATH)
print(" -", PARAMS_JSON)
print(" -", PARAMS_MD)
print(" -", MANIFEST_JSON)